In [0]:
%python
# Databricks Notebook Source
# ==============================================================================
# PIPELINE SILVER & EDA AUTOMATIZADO - GIVE ME SOME CREDIT
# Arquitetura Medallion: Bronze -> Silver
# ==============================================================================

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType, IntegerType

# 1. CONFIGURAÇÃO DE AMBIENTE E PARÂMETROS
CATALOG_NAME = "credito_prd"
SCHEMA_BRONZE = "bronze"
SCHEMA_SILVER = "silver"
TABLE_NAME = "give_me_some_credit"

source_table = f"{CATALOG_NAME}.{SCHEMA_BRONZE}.{TABLE_NAME}_raw"
target_table = f"{CATALOG_NAME}.{SCHEMA_SILVER}.{TABLE_NAME}"

# Garantir existência do schema silver
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG_NAME}.{SCHEMA_SILVER}")

print(f">>> Lendo dados da Bronze: {source_table}")
df_raw = spark.table(source_table)

# ==============================================================================
# 2. DATA CLEANING & STANDARDIZATION (PADRÃO DE MERCADO)
# ==============================================================================

# 2.1 Padronização de nomes de colunas (snake_case)
column_mapping = {
    "customer_id": "customer_id",
    "SeriousDlqin2yrs": "target_default_2yrs",
    "RevolvingUtilizationOfUnsecuredLines": "revolving_utilization_unsecured",
    "age": "age",
    "NumberOfTime30-59DaysPastDueNotWorse": "num_times_30_59_days_late",
    "DebtRatio": "debt_ratio",
    "MonthlyIncome": "monthly_income",
    "NumberOfOpenCreditLinesAndLoans": "num_open_credit_lines_and_loans",
    "NumberOfTimes90DaysLate": "num_times_90_days_late",
    "NumberRealEstateLoansOrLines": "num_real_estate_loans_or_lines",
    "NumberOfTime60-89DaysPastDueNotWorse": "num_times_60_89_days_late",
    "NumberOfDependents": "num_dependents"
}

df_renamed = df_raw
for old_col, new_col in column_mapping.items():
    if old_col in df_renamed.columns:
        df_renamed = df_renamed.withColumnRenamed(old_col, new_col)

# 2.2 Desduplicação com base na chave de negócio
df_dedup = df_renamed.dropDuplicates(subset=["customer_id"])

# 2.3 Tratamento de Tipagem, Nulos e Regras de Sanidade
# - Flags de qualidade (Quarentena lógica sem descarte prematuro)
# - Missing value imputation / flags
df_silver_transformed = (
    df_dedup
    .withColumn("age", F.when((F.col("age") < 18) | (F.col("age") > 115), None).otherwise(F.col("age")))
    .withColumn("is_monthly_income_null", F.when(F.col("monthly_income").isNull(), 1).otherwise(0))
    .withColumn("num_dependents", F.coalesce(F.col("num_dependents").cast(IntegerType()), F.lit(0)))
    .withColumn(
        "has_delinquency_outlier",
        F.when(
            (F.col("num_times_30_59_days_late") >= 96) | 
            (F.col("num_times_60_89_days_late") >= 96) | 
            (F.col("num_times_90_days_late") >= 96), 
            1
        ).otherwise(0)
    )
    # Metadados de linhagem técnica
    .withColumn("_ingestion_bronze_ref", F.lit(source_table))
    .withColumn("_silver_processed_at", F.current_timestamp())
)

# ==============================================================================
# 3. EDA AUTOMATIZADO (EXPLORATORY DATA ANALYSIS)
# ==============================================================================
print("\n" + "="*80)
print("RELATÓRIO DE DATA QUALITY & EDA (SILVER LAYER)")
print("="*80)

total_bronze = df_raw.count()
total_silver = df_silver_transformed.count()
duplicatas_removidas = total_bronze - total_silver

print(f"Total registros na Bronze: {total_bronze:,}")
print(f"Total registros na Silver: {total_silver:,}")
print(f"Linhas duplicadas removidas: {duplicatas_removidas:,}")

# 3.1 Checagem de Nulos e Completude por Coluna
null_counts_exprs = [F.sum(F.when(F.col(c).isNull(), 1).otherwise(0)).alias(c) for c in df_silver_transformed.columns]
null_counts = df_silver_transformed.agg(*null_counts_exprs).collect()[0].asDict()

print("\n--- COMPLETUDE DOS DADOS (NULOS POR ATRIBUTO) ---")
for col_name, missing_count in null_counts.items():
    pct_missing = (missing_count / total_silver) * 100
    print(f"• {col_name:35} | Faltantes: {missing_count:8,} | {pct_missing:6.2f}%")

# 3.2 Distribuição da Variável Alvo (Taxa de Inadimplência)
if "target_default_2yrs" in df_silver_transformed.columns:
    print("\n--- DISTRIBUIÇÃO DA CLASSE (DEFAULT RATE) ---")
    df_silver_transformed.groupBy("target_default_2yrs").agg(
        F.count("*").alias("volume"),
        F.round(F.count("*") / total_silver * 100, 2).alias("percentual")
    ).show()

# 3.3 Resumo Numérico (Percentis 25%, 50%, 75%, Média e Desvio)
numeric_cols = [
    "age", "monthly_income", "debt_ratio", 
    "revolving_utilization_unsecured", "num_open_credit_lines_and_loans"
]
print("\n--- MÉTRICAS ESTATÍSTICAS DOS ATRIBUTOS CONTÍNUOS ---")
df_silver_transformed.select(numeric_cols).summary("mean", "stddev", "min", "25%", "50%", "75%", "max").show()

# # ==============================================================================
# # 4. PERSISTÊNCIA EM DELTA TABLE (UNITY CATALOG)
# # ==============================================================================

# Por ter sido criada via streaming no DLT, o Databricks bloqueia qualquer sobrescrita manual com write.mode("overwrite"), gerando o erro #STREAMING_TABLE_OPERATION_NOT_ALLOWED.
# print(f">>> Gravando tabela Silver: {target_table}")
# isso será feito na celula abaixo

# (
#     df_silver_transformed.write
#     .format("delta")
#     .mode("overwrite")
#     .option("overwriteSchema", "true")
#     .saveAsTable(target_table)
# )

print(f"[OK] Carga finalizada com sucesso em: {target_table}")

# 5. VISUALIZAÇÃO INTERATIVA DO DATASET HIGIENIZADO
# (Use o botão '+' no cabeçalho da tabela gerada abaixo para abrir gráficos nativos)
display(spark.table(target_table).limit(10))

In [0]:
%python
# ==============================================================================
# VISUALIZAÇÃO GRÁFICA INTERATIVA (SILVER)
# Utiliza as variáveis geradas na célula anterior: df_silver_transformed e total_silver
# ==============================================================================

# Visão 1: Gráfico de Barras / Pizza da Distribuição da Variável Alvo (Target)
display(
    df_silver_transformed.groupBy("target_default_2yrs")
    .count()
    .withColumn("percentual", (F.col("count") / total_silver) * 100)
)

# Visão 2: Idade vs Target (descomente para usar)
# display(df_silver_transformed.select("age", "target_default_2yrs"))

# Visão 3: Relação entre Atrasos e Inadimplência (descomente para usar)
# display(df_silver_transformed.groupBy("num_times_30_59_days_late", "target_default_2yrs").count())

In [0]:
%python
# usando a biblioteca matplolib
# Caso queira gerar gráficos via código sem usar a interface do display(), a alternativa no Python é usar bibliotecas gráficas estáticas (como matplotlib ou seaborn):


import matplotlib.pyplot as plt

# Converte os dados agregados para Pandas e plota direto no notebook
pdf = (
    df_silver_transformed.groupBy("target_default_2yrs")
    .count()
    .toPandas()
)




plt.figure(figsize=(6, 4))
plt.bar(pdf["target_default_2yrs"].astype(str), pdf["count"], color=["#1f77b4", "#d62728"])
plt.title("Distribuição do Target")
plt.xlabel("Inadimplente (1) vs Adimplente (0)")
plt.ylabel("Contagem")
plt.show()

In [0]:
%python
from pyspark.sql import functions as F

display(
    df_silver_transformed
    .withColumn(
        "status_cliente",
        F.when(F.col("target_default_2yrs") == 1, "Inadimplente (1)")
         .otherwise("Adimplente (0)")
    )
    .groupBy("status_cliente")
    .count()
)

plt.figure(figsize=(6, 4))
plt.bar(pdf["target_default_2yrs"].astype(str), pdf["count"], color=["#1f77b4", "#d62728"])
plt.title("Distribuição do Target")
plt.xlabel("Inadimplente (1) vs Adimplente (0)")
plt.ylabel("Contagem")
plt.show()



In [0]:
%python
import matplotlib.pyplot as plt
from pyspark.sql import functions as F

# 1. Agregação e conversão direta para Pandas
pdf = (
    df_silver_transformed
    .withColumn(
        "status_cliente",
        F.when(F.col("target_default_2yrs") == 1, "Inadimplente (1)")
         .otherwise("Adimplente (0)")
    )
    .groupBy("status_cliente")
    .count()
    .toPandas()
)

# 2. Construção do gráfico com Matplotlib
fig, ax = plt.subplots(figsize=(6, 4))
bars = ax.bar(pdf["status_cliente"], pdf["count"], color=["#1f77b4", "#d62728"])

# 3. Exibe os valores dentro de cada barra (formatado com separador de milhar)
ax.bar_label(bars, fmt="{:,.0f}", label_type="center", color="white", fontweight="bold", fontsize=11)

ax.set_title("Distribuição do Target", fontsize=12, fontweight="bold")
ax.set_xlabel("Status do Cliente")
ax.set_ylabel("Contagem")
plt.tight_layout()
plt.show()